In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
from monai.transforms import LoadImage, EnsureChannelFirst, Orientation, Compose
from monai.bundle import ConfigParser, download
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
import seaborn as sns
from ipywidgets import interact

from monai.transforms import Resize



In [2]:
def show_scrollable_image(image, cmap="nipy_spectral"):
    """
    Displays a 3D array as a scrollable series of slices.
    
    Parameters:
    - image (numpy.ndarray): A 3D numpy array (e.g., (H, W, D)) representing the image.
    """
    
    # Define a function to display each slice
    def display_slice(slice_index):
        plt.figure(figsize=(8, 8))
        plt.imshow(image[:, :, slice_index], cmap=cmap)
        plt.colorbar(label='HU')
        plt.axis("off")
        plt.title(f"Slice {slice_index + 1} / {image.shape[2]}")
        plt.show()
    
    # Create an interactive slider to scroll through slices
    interact(display_slice, slice_index=(0, image.shape[2] - 1))

In [3]:
import os

# Define the path to the main directory
testing_subset_path = '../data/Inputs/takotsubo_cases/testing_subset/'

min_slices = float('inf')

# Loop through all folders in the directory
for folder_name in sorted(os.listdir(testing_subset_path)):
    folder_path = os.path.join(testing_subset_path, folder_name)
    
    # Check if the path is a directory
    if os.path.isdir(folder_path):
        print(folder_name)
    
        dicom_files_folder = os.path.join(folder_path, "dicom", "EXP00000")
        print(dicom_files_folder)
        num_slices = len(os.listdir(dicom_files_folder))
        min_slices = min(min_slices, num_slices)
        print(num_slices)
        
        # for dicom_file in sorted(os.listdir(dicom_files_folder)):
        #     print(f"    {dicom_file}")
print(f"Min_slices: {min_slices}")  

JNMP 02291631
../data/Inputs/takotsubo_cases/testing_subset/JNMP 02291631/dicom/EXP00000
151
JNMP 2nd scan
../data/Inputs/takotsubo_cases/testing_subset/JNMP 2nd scan/dicom/EXP00000
96
JOS 3812401
../data/Inputs/takotsubo_cases/testing_subset/JOS 3812401/dicom/EXP00000
142
LET 04456356
../data/Inputs/takotsubo_cases/testing_subset/LET 04456356/dicom/EXP00000
982
LMA 03884426
../data/Inputs/takotsubo_cases/testing_subset/LMA 03884426/dicom/EXP00000
334
LMC 03629185
../data/Inputs/takotsubo_cases/testing_subset/LMC 03629185/dicom/EXP00000
210
LMD 22194237
../data/Inputs/takotsubo_cases/testing_subset/LMD 22194237/dicom/EXP00000
344
LW 12109724
../data/Inputs/takotsubo_cases/testing_subset/LW 12109724/dicom/EXP00000
235
MC 03887643
../data/Inputs/takotsubo_cases/testing_subset/MC 03887643/dicom/EXP00000
159
MDW 2286750
../data/Inputs/takotsubo_cases/testing_subset/MDW 2286750/dicom/EXP00000
111
MG 22519714
../data/Inputs/takotsubo_cases/testing_subset/MG 22519714/dicom/EXP00000
92
MGR 025

In [4]:
# dicom_file_path = 'data/normal_cases/AG 519880_37F/DICOM/EXP00000/'
# dicom_file_path = 'data/normal_cases/ASJ 05633540_68F/DICOM/EXP00000/'
# dicom_file_path = '../data/takotsubo_cases/testing_subset/JNMP 02291631/DICOM/EXP00000/'
# dicom_file_path = '../data/takotsubo_cases/testing_subset/JOS 3812401/DICOM/EXP00000/'

input_folder = "../data/Inputs/takotsubo_cases/testing_subset/LET 04456356"
dicom_file_path = os.path.join(input_folder, "DICOM", "EXP00000")

output_folder = "../data/Outputs/takotsubo_cases/testing_subset/LET 04456356"

image_loader = LoadImage(image_only=True)
original_input_image = image_loader(dicom_file_path)

original_input_image.meta

{'00200037': {'vr': 'DS', 'Value': [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]},
 '00200032': {'vr': 'DS', 'Value': [-195.7, -175.7, 50.0]},
 '00280030': {'vr': 'DS', 'Value': [0.806641, 0.806641, 0.625]},
 'spacing': array([0.806641, 0.806641, 0.625   ]),
 'lastImagePositionPatient': array([-195.7  , -175.7  , -563.125]),
 spatial_shape: (512, 512, 982),
 space: RAS,
 original_affine: array([[ -0.806641,   0.      ,   0.      , 195.7     ],
        [  0.      ,  -0.806641,   0.      , 175.7     ],
        [  0.      ,   0.      ,  -0.625   ,  50.      ],
        [  0.      ,   0.      ,   0.      ,   1.      ]]),
 affine: tensor([[ -0.8066,   0.0000,   0.0000, 195.7000],
         [  0.0000,  -0.8066,   0.0000, 175.7000],
         [  0.0000,   0.0000,  -0.6250,  50.0000],
         [  0.0000,   0.0000,   0.0000,   1.0000]], dtype=torch.float64),
 original_channel_dim: nan,
 'filename_or_obj': '../data/Inputs/takotsubo_cases/testing_subset/LET 04456356/DICOM/EXP00000'}

In [5]:
show_scrollable_image(original_input_image, cmap="grey")

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [6]:
import nibabel as nib
from totalsegmentator.python_api import totalsegmentator


# option 1: provide input and output as file paths
# output_directory = "../data/segment_eval_output_takotsubo_case2_bodyseg"
# output_image = totalsegmentator(dicom_file_path, output_folder, license_number="aca_BWYHC6UQQFDU8A", task="heartchambers_highres", body_seg=True)

# option 2: provide input and output as nifti image objects
# input_img = nib.load(dicom_file_path)
# output_img = totalsegmentator(input_img)
# print(output_img)
# nib.save(output_img, data_directory)

In [7]:
# nib.save(output_image, f'{output_folder}/output_image.nii.gz')

In [7]:
full_image = image_loader(f"{output_folder}/output_image.nii.gz")
show_scrollable_image(full_image)

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [8]:
# Load the image
left_ventricle_img = image_loader(f"{output_folder}/heart_ventricle_left.nii.gz")

# Check the image type and shape
print(type(left_ventricle_img))
print(left_ventricle_img.shape)

show_scrollable_image(left_ventricle_img)

<class 'monai.data.meta_tensor.MetaTensor'>
torch.Size([512, 512, 982])


interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [23]:
# def resize_to_target_slices(image, heart_mask, target_slices=80):
    
#     z_indices = np.any(heart_mask.get_fdata() > 0, axis=(0, 1))
#     heart_slices = np.where(z_indices)[0]
#     print(heart_slices)
    
#     center_slice = (heart_slices[0] + heart_slices[-1]) // 2
#     start_slice = max(0, center_slice - target_slices // 2)
#     end_slice = start_slice + target_slices
#     return image[:, :, start_slice:end_slice]
    
    
def resize_to_target_slices(image, heart_mask, target_slices=128):
    
    z_indices = np.any(heart_mask > 0, axis=(0, 1))
    heart_slices = np.where(z_indices)[0]
    print(heart_slices)
    
    start_slice = heart_slices[0]
    end_slice = heart_slices[-1] + 1
    sliced_image = image[:, :, start_slice:end_slice].as_tensor()
    
    sliced_image = sliced_image.unsqueeze(0)  # Shape becomes (1, H, W, Z)

    
    print(sliced_image.shape)
    
    resizer = Resize(spatial_size=(512,512,target_slices))
    resized_image = resizer(sliced_image)
    
    resized_image = resized_image.squeeze(0)
    return resized_image
    
    
resized_image = resize_to_target_slices(full_image, full_image)




[556 557 558 559 560 561 562 563 564 565 566 567 568 569 570 571 572 573
 574 575 576 577 578 579 580 581 582 583 584 585 586 587 588 589 590 591
 592 593 594 595 596 597 598 599 600 601 602 603 604 605 606 607 608 609
 610 611 612 613 614 615 616 617 618 619 620 621 622 623 624 625 626 627
 628 629 630 631 632 633 634 635 636 637 638 639 640 641 642 643 644 645
 646 647 648 649 650 651 652 653 654 655 656 657 658 659 660 661 662 663
 664 665 666 667 668 669 670 671 672 673 674 675 676 677 678 679 680 681
 682 683 684 685 686 687 688 689 690 691 692 693 694 695 696 697 698 699
 700 701 702 703 704 705 706 707 708 709 710 711 712 713 714 715 716 717
 718 719 720 721 722 723 724 725 726 727 728 729 730 731 732 733 734 735
 736 737 738 739 740 741 742 743 744 745 746 747 748 749 750 751 752 753
 754 755 756 757 758 759 760 761 762 763 764 765 766 767 768 769 770 771
 772 773 774 775 776 777 778 779 780 781 782 783 784 785 786 787 788 789
 790 791 792 793 794 795 796 797 798 799 800 801 80

In [24]:
print(resized_image.shape)


torch.Size([512, 512, 128])


In [25]:
show_scrollable_image(resized_image, cmap="grey")

interactive(children=(IntSlider(value=63, description='slice_index', max=127), Output()), _dom_classes=('widge…

In [26]:
def align_segmentation_to_image(segmentation):
    """
    Aligns the TotalSegmentator output (segmentation) with the original image
    by flipping the z-axis of the segmentation mask.
    
    Parameters:
        image (torch.Tensor or np.ndarray): Original image (H x W x D).
        segmentation (torch.Tensor or np.ndarray): TotalSegmentator output (H x W x D).
        
    Returns:
        torch.Tensor: Segmentation aligned with the image.
    """
   
    # Flip along the z-axis (last dimension)
    aligned_segmentation = torch.flip(segmentation, dims=[-1])
    
    return aligned_segmentation

# Example usage
aligned_mask = align_segmentation_to_image(full_image)



In [27]:
show_scrollable_image(aligned_mask)

interactive(children=(IntSlider(value=490, description='slice_index', max=981), Output()), _dom_classes=('widg…

In [28]:
original_image_resized = resize_to_target_slices(original_input_image, aligned_mask)

[138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155
 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173
 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191
 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209
 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227
 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245
 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263
 264 265 266 267 268 269 270 271 272 273 274 275 276 277 278 279 280 281
 282 283 284 285 286 287 288 289 290 291 292 293 294 295 296 297 298 299
 300 301 302 303 304 305 306 307 308 309 310 311 312 313 314 315 316 317
 318 319 320 321 322 323 324 325 326 327 328 329 330 331 332 333 334 335
 336 337 338 339 340 341 342 343 344 345 346 347 348 349 350 351 352 353
 354 355 356 357 358 359 360 361 362 363 364 365 366 367 368 369 370 371
 372 373 374 375 376 377 378 379 380 381 382 383 38

In [30]:
show_scrollable_image(original_image_resized, cmap="grey")

interactive(children=(IntSlider(value=63, description='slice_index', max=127), Output()), _dom_classes=('widge…

Bad pipe message: %s [b'@\xb4a\xb1\x1f\x98\x80\xb6\xde\x89\xbf\xef%iJ\rOk \xc2\x91\xef\xef\x88\xb3\x1bL^kj\'c7y\xc6\xd0\x96BI\xd1S\x1d;fx\x88\x17l\x8b\xcd-\x00 \xda\xda\x13\x01\x13\x02\x13\x03\xc0+\xc0/\xc0,\xc00\xcc\xa9\xcc\xa8\xc0\x13\xc0\x14\x00\x9c\x00\x9d\x00/\x005\x01\x00\x06\x93\x1a\x1a\x00\x00\x00-\x00\x02\x01\x01Di\x00\x05\x00\x03\x02h2\xfe\r\x01\x1a\x00\x00\x01\x00\x01\x93\x00 "@t\x12\xd0\xd9\xbbUN\x1a\x15\xd6;\xa1\xbf\xa1\x1d\xff\xfe\x84\\\xf7\x96\xa56./\x12\xeb\x1cdF\x00\xf0\x95\x01\xe6\xe5\x16o\x15\xefU\x9c\xc0\xda\xfb}>JV\xfa\x85]\xf7\xdc&X.\xe3Y \xa3f\n\x08\x93\xa0\xe4H\xado\x92\xf9o\xb4YD}M\x1d\x83H\'\xc09\xb4z2\xf1L\xf4\xba\xa1\x007\\\x89p\xf6\xb0zi\xfe\x87\xeb']
Bad pipe message: %s [b'\x9e\xa2\x07\xa0\xd5R\xeb\xa3\xd7\xb0Ku\xe7\x85\x1b\x89 Y\xe4a9\xde\x06x\xe2V\x1d\x05s<\x1d\xae\xa1\xfb\x1bK/\xb1\xc7\xa5XX\xe7\x96\xac\xe4 \x0f\x16L\xec\xbf6\xf3z\xdd\xcc\xa79DK\xc3\xd3\xe9{\x05\xfak.\x19-\xd3\x81K\xa4\xc3\x86\xbe\xd2\xa2\x9c&\xbcw,\xca\xf1\x18\xba\xbc\xff\xc9\x04\xa8\